# OcuWeave–LQNet · frozen exploratory development
Marchel Andrian Shevchenko · Data Sorcerers · 2026 · software v0.8.2.

This clean launcher prepares a **new** experiment. It pins a release, preserves historical test artifacts, provisionally groups development near-duplicate candidates, and locks nine runs. All quality-flagged images remain eligible. Patient identity and clinical annotations remain unverified. No U-Net, pretrained weights or EMR.

CPU smoke checks cannot unlock GPU training. Record current allocation errors as observations; a fresh template does not establish GPU availability or completed results.

For an existing frozen suite, follow `docs/protocols/exploratory_training.md` instead of creating a replacement manifest or rerunning setup into a new suite. The executed 31 October session used v0.8.1; its source contract remains unchanged by this launcher update.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
from google.colab import drive
import os,sys,json,subprocess,hashlib,shutil
from IPython.display import display,Markdown,Image
PROJECT=Path('/content/drive/MyDrive/OcuWeave — Datasets and experiments')
if not PROJECT.exists():drive.mount('/content/drive')
assert PROJECT.is_dir()
# Resolve the source once; every subsequent artifact records this exact commit.
REVISION=os.environ.get('OCUWEAVE_EXPLORATORY_REVISION','v0.8.2')
REPO=Path('/content/ocuweave-exploratory-v082')
if not REPO.exists():subprocess.run(['git','clone','https://github.com/chenko-the-sorcerers/ocuweave.git',str(REPO)],check=True,capture_output=True)
subprocess.run(['git','fetch','origin','--tags'],cwd=REPO,check=True,capture_output=True)
subprocess.run(['git','checkout','--detach',REVISION],cwd=REPO,check=True,capture_output=True)
COMMIT=subprocess.check_output(['git','rev-parse','HEAD'],cwd=REPO,text=True).strip()
assert not subprocess.check_output(['git','status','--porcelain'],cwd=REPO,text=True).strip()
STAMP=datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
SESSION=PROJECT/'02 — Audits/exploratory-v0.8.2'/STAMP
SESSION.mkdir(parents=True,exist_ok=False)
ENV=dict(os.environ,PYTHONPATH=str(REPO/'src'),OMP_NUM_THREADS='2',OPENBLAS_NUM_THREADS='2',MKL_NUM_THREADS='2',MPLBACKEND='Agg')
def run(args,name):
    with (SESSION/name).open('x') as log:
        p=subprocess.run(args,cwd=REPO,env=ENV,stdout=log,stderr=subprocess.STDOUT)
    print(name,'exit',p.returncode,(SESSION/name).read_text()[-1800:])
    p.check_returncode()
def digest(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as f:
        for block in iter(lambda:f.read(1024*1024),b''):h.update(block)
    return h.hexdigest()
import torch
print({'commit':COMMIT,'torch':str(torch.__version__),'cuda':torch.cuda.is_available(),'evidence':str(SESSION)})


## 1. Verify software
Synthetic tests cover grouping, test preservation, gradients, checkpoint reload and CPU/CUDA gate separation. These are implementation checks, not model-performance results.

In [ ]:
run([sys.executable,'-m','pip','install','-e','.', '--no-deps'],'install.log')
run(['cmake','-S','.','-B','build','-DBUILD_TESTING=ON'],'cmake.log')
run(['cmake','--build','build'],'build.log')
run(['ctest','--test-dir','build','--output-on-failure'],'ctest.log')
ENV['OCUWEAVE_NATIVE_LIBRARY']=str(REPO/'build/libocuweave_native.so')
run([sys.executable,'-m','unittest','discover','-s','tests','-v'],'python-tests.log')
import ast,nbformat
nb=nbformat.read(REPO/'notebooks/OcuWeave_Exploratory_Colab.ipynb',as_version=4);nbformat.validate(nb)
for c in nb.cells:
    if c.cell_type=='code':ast.parse(c.source)
print('Notebook syntax/schema checked')


## 2. Freeze a new development manifest
No raw file, mask, old split or test result is overwritten. Historical quality flags remain review priorities; all flagged development images stay eligible. The candidate family is provisional, not a verified patient group.

In [ ]:
SOURCE=PROJECT/'01 — Raw datasets/OTFID_v3'
ORIGINAL_ARCHIVE=SOURCE/'Ocular_Toxoplasmosis_Data_V3.zip'
OLD_MANIFEST=PROJECT/'02 — Audits/otfid-v3-binary-v2-seed42-v0.2.0/manifest.json'
PILOT=PROJECT/'04 — Runs/v0.2.0/pilot-seed42-20epochs'
EXPECTED={OLD_MANIFEST:'b8cc4f42779d57ad5ad0fa18704730edaf62650d5594996ffab1cc6d25cb980f',PILOT/'query-test.json':'42a1a5aac443816cb4ed08c87411249f03ef8e283851674c389ca650b90d8f42',PILOT/'fcn-test.json':'0a94ce5aa157d0600ea8ea41a24797e4f3d3b38a8c3887877145f8dc06191f2c'}
assert all(digest(path)==value for path,value in EXPECTED.items())
PROTECTED=[OLD_MANIFEST,ORIGINAL_ARCHIVE,*PILOT.glob('*-test.json'),*PILOT.glob('*/best.pt')]
BEFORE={str(p.relative_to(PROJECT)):digest(p) for p in PROTECTED}
(SESSION/'protected-before.json').write_text(json.dumps(BEFORE,indent=2))
ARCHIVE=Path('/content/ocuweave-exploratory-otfid.zip')
if not ARCHIVE.exists():shutil.copyfile(ORIGINAL_ARCHIVE,ARCHIVE)
assert digest(ARCHIVE)==json.loads((SOURCE/'download_manifest.json').read_text())['sha256']
AUDIT=PROJECT/'02 — Audits/cleaning-review-v0.5.0/20261031T061228Z/review'
FROZEN=SESSION/'development-manifest'
run([sys.executable,'scripts/freeze_development.py','--manifest',str(OLD_MANIFEST),'--duplicates',str(AUDIT/'otfid/near-duplicates.json'),'--audit-summary',str(AUDIT/'summary.json'),'--quality-csv',str(AUDIT/'otfid/image-quality-review.csv'),'--output',str(FROZEN)],'freeze.log')
MANIFEST=FROZEN/'manifest.json'
DECISION=json.loads((FROZEN/'decision.json').read_text())
print({k:DECISION[k] for k in ('counts_before','counts_after','flagged_development_retained','test_rows_unchanged','manifest_sha256')})


## 3. Lock all nine runs
Same 20 epochs, batch 8, loss, optimizer, augmentation and sample membership. Each run starts from random weights. Configuration/source/manifest changes require a new suite. Original-grid validation is distinct from historical pilot metrics.

In [ ]:
SUITE=PROJECT/'04 — Runs/v0.8.2'/STAMP/'resolution-detail-seeds'
COMMON=['--manifest',str(MANIFEST),'--archive',str(ARCHIVE)]
run([sys.executable,'scripts/run_exploratory.py','register',*COMMON,'--output',str(SUITE)],'register.log')
CONTRACT=json.loads((SUITE/'locked-protocol.json').read_text())
print({'planned_runs':len(CONTRACT['configs']),'optimizer_steps_each':CONTRACT['optimizer_steps_per_run'],'counts':CONTRACT['split_counts'],'source_commit':CONTRACT['source_commit']})


## 4. Smoke test and paired-image inspection
Three batches per variant. Loss/gradient checks and checkpoint readback are pipeline evidence only. When CUDA is absent, an explicitly labelled CPU check runs; it cannot unlock the research suite. Inspect reference overlays before interpreting training.

In [ ]:
DEVICE='cuda' if torch.cuda.is_available() else 'cpu'
SMOKE=SESSION/(DEVICE+'-smoke')
run([sys.executable,'scripts/run_exploratory.py','smoke',*COMMON,'--output',str(SMOKE),'--device',DEVICE],'smoke.log')
SMOKE_REPORT=json.loads((SMOKE/'smoke.json').read_text())
print({'status':SMOKE_REPORT['status'],'device':DEVICE,'gpu_gate_passed':SMOKE_REPORT['gpu_gate_passed']})
for p in sorted(SMOKE.glob('*-pairs.png')):display(Image(filename=str(p),width=700))


## 5. Execute only after the GPU gate
Seed order: 42, 123, 2026; variants Query 192, Query 384, OcuWeave–LQNet 384. No EMR and no historical test evaluation. Incomplete runs remain visible and are never counted as complete.

In [ ]:
if SMOKE_REPORT['gpu_gate_passed']:
    run([sys.executable,'scripts/run_exploratory.py','execute',*COMMON,'--output',str(SUITE),'--smoke-report',str(SMOKE/'smoke.json')],'suite.log')
    TRAINING_STATUS='See verified completion receipts'
else:
    TRAINING_STATUS='Blocked: CUDA unavailable; no passed GPU smoke gate. See the observed runtime availability recorded for this session.'
REGISTER=json.loads((SUITE/'summary.json').read_text())
COMPLETED=[r for r in REGISTER['runs'] if r['status']=='completed']
display(Markdown((SUITE/'README.md').read_text()))
print(TRAINING_STATUS)


## 6. Preserve evidence and report observed status
Validation records include lesion/small-lesion Dice, IoU, healthy false-positive area, time, memory and step counts. Only complete, hash-verified runs enter comparisons; absent results remain absent.

In [ ]:
AFTER={str(p.relative_to(PROJECT)):digest(p) for p in PROTECTED}
assert AFTER==BEFORE
(SESSION/'protected-after.json').write_text(json.dumps(AFTER,indent=2))
STATUS={'source_commit':COMMIT,'manifest_sha256':digest(MANIFEST),'locked_protocol_sha256':digest(SUITE/'locked-protocol.json'),'split_counts':CONTRACT['split_counts'],'flagged_development_retained':DECISION['flagged_development_retained'],'optimizer_steps_per_run':CONTRACT['optimizer_steps_per_run'],'smoke_device':DEVICE,'gpu_smoke_passed':SMOKE_REPORT['gpu_gate_passed'],'completed_runs':len(COMPLETED),'planned_runs':9,'status':TRAINING_STATUS,'test_evaluated':False,'protected_hashes_unchanged':True,'patient_identity_verified':False,'emr_used':False}
(SESSION/'session-status.json').write_text(json.dumps(STATUS,indent=2))
SNAPSHOT=PROJECT/'05 — Research documentation/v0.8.2'/STAMP
run([sys.executable,'scripts/archive_research_record.py','--project',str(PROJECT),'--output',str(SNAPSHOT)],'documentation.log')
print(json.dumps(STATUS,indent=2))
